In [0]:
%run "../_shared/02_football_data_org_client"

In [0]:
import logging
import os


logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(levelname)s | "
        "%(message)s"
    ),
)

logger = logging.getLogger(__name__)

In [0]:
TOP_5_COMPETITIONS = {
    "PL": "premier_league",
    "PD": "la_liga",
    "BL1": "bundesliga",
    "SA": "serie_a",
    "FL1": "ligue_1",
}


BASE_URL = "https://api.football-data.org/v4"

RAW_BASE_PATH = (
    "/Volumes/football_data_platform/raw/source_files/"
    "football_data_org/standings"
)


API_TOKEN = dbutils.secrets.get(
    scope="football-api",
    key="api-token",
)

In [0]:
def validate_standings_payload(
    payload: dict,
    expected_competition_code: str,
) -> None:
    """
    Validate basic business-level properties
    of a standings response.
    """

    competition = payload.get(
        "competition",
        {},
    )

    returned_competition_code = competition.get(
        "code"
    )

    if returned_competition_code != expected_competition_code:
        raise FootballDataOrgError(
            "Unexpected competition returned by API. "
            f"Requested '{expected_competition_code}', "
            f"received '{returned_competition_code}'."
        )

    season = payload.get(
        "season"
    )

    if not isinstance(season, dict):
        raise FootballDataOrgError(
            "Expected 'season' to be an object."
        )

    standings = payload.get(
        "standings"
    )

    if not isinstance(standings, list):
        raise FootballDataOrgError(
            "Expected 'standings' to be a list."
        )

    if not standings:
        raise FootballDataOrgError(
            "football-data.org returned an empty standings list "
            f"for competition '{expected_competition_code}'."
        )

    for standing in standings:

        if not isinstance(standing, dict):
            raise FootballDataOrgError(
                "Expected each standings entry to be an object."
            )

        standing_type = standing.get(
            "type"
        )

        if not standing_type:
            raise FootballDataOrgError(
                "Standing entry does not contain a type."
            )

        table = standing.get(
            "table"
        )

        if not isinstance(table, list):
            raise FootballDataOrgError(
                "Expected standing 'table' to be a list "
                f"for type '{standing_type}'."
            )

        if not table:
            raise FootballDataOrgError(
                "Standing table is empty "
                f"for type '{standing_type}'."
            )

In [0]:
def extract_standings(
    client: FootballDataOrgClient,
    competitions: dict[str, str],
    output_directory: str,
) -> dict[str, int]:
    """
    Extract current standings for configured competitions.

    Each competition response is stored as one raw JSON file.

    Existing files are skipped to keep reruns idempotent
    during the initial ingestion phase.
    """

    extracted_count = 0
    skipped_count = 0
    failed_count = 0

    for competition_code, competition_name in competitions.items():

        file_name = (
            f"{competition_code.lower()}_"
            f"{competition_name}.json"
        )

        file_path = os.path.join(
            output_directory,
            file_name,
        )

        if os.path.exists(file_path):

            logger.info(
                "Skipping existing standings data: %s",
                competition_code,
            )

            skipped_count += 1
            continue

        logger.info(
            "Extracting standings: %s",
            competition_code,
        )

        try:

            payload = client.get_competition_standings(
                competition_code=competition_code,
            )

            validate_standings_payload(
                payload=payload,
                expected_competition_code=competition_code,
            )

            write_json_atomic(
                data=payload,
                file_path=file_path,
            )

            extracted_count += 1

            standings = payload.get(
                "standings",
                [],
            )

            standing_types = [
                standing.get("type")
                for standing in standings
            ]

            logger.info(
                "Saved standings data: "
                "%s | tables=%s | types=%s",
                competition_code,
                len(standings),
                standing_types,
            )

        except FootballDataOrgRateLimitError:

            logger.exception(
                "Rate limit reached while extracting %s. "
                "Stopping ingestion.",
                competition_code,
            )

            raise

        except FootballDataOrgError:

            failed_count += 1

            logger.exception(
                "Failed to extract standings: %s",
                competition_code,
            )

        except Exception:

            failed_count += 1

            logger.exception(
                "Unexpected error while extracting standings: %s",
                competition_code,
            )

    return {
        "extracted": extracted_count,
        "skipped": skipped_count,
        "failed": failed_count,
    }

In [0]:
with FootballDataOrgClient(
    api_token=API_TOKEN,
    base_url=BASE_URL,
    min_request_interval_seconds=6.5,
    timeout_seconds=30,
) as client:

    ingestion_summary = extract_standings(
        client=client,
        competitions=TOP_5_COMPETITIONS,
        output_directory=RAW_BASE_PATH,
    )

In [0]:
logger.info(
    "Standings ingestion completed. "
    "Extracted: %s | Skipped: %s | Failed: %s",
    ingestion_summary["extracted"],
    ingestion_summary["skipped"],
    ingestion_summary["failed"],
)

if ingestion_summary["failed"] > 0:
    raise RuntimeError(
        "Standings ingestion completed with failures. "
        f"Failed extractions: "
        f"{ingestion_summary['failed']}"
    )

ingestion_summary